# Unsupervised Learning Defect Detector（整理版）

SEM 晶粒微結構影像的無監督缺陷偵測。

- **Part A**：PCA + K-means 微結構分群 → cluster map
- **Part B（主角）**：CNN Autoencoder 重建誤差 → anomaly heatmap + defect mask

> 本檔為 `unsupervised_learning_defect_detector.ipynb` 的整理版：合併重複 cell、
> 統一成單一套分群、將所有參數集中到下方 CONFIG。變更明細見 `CHANGELOG.md`。


## 0. 全域設定（CONFIG）

所有可調參數集中於此,方便調參與復現。

In [ ]:
# ===== 資料/路徑 =====
from pathlib import Path

DATA_ROOT = Path("/content/grainsize_dataset")   # dataset 解壓後的資料夾
IMG_EXT   = [".jpg", ".png", ".jpeg", ".tif", ".tiff"]

# ===== 前處理 / 切 patch =====
RESIZE_TO         = (512, 512)   # 統一影像尺寸
PATCH_SIZE        = 64
STRIDE            = 32           # < PATCH_SIZE → patch 之間 50% 重疊
MAX_PATCHES_TOTAL = 80000        # 記憶體上限保護（Colab RAM 有限）

# ===== Part A：PCA + K-means =====
PCA_MAX_SAMPLES = 30000          # 做分群時最多取多少 patch（省 RAM）
PCA_COMPONENTS  = 50
N_CLUSTERS      = 5

# ===== Part B：Autoencoder =====
AE_MAX_TRAIN = 20000             # 訓練用 patch 數
AE_EPOCHS    = 15
AE_BATCH     = 256
AE_LR        = 1e-3

# ===== 異常門檻 =====
ANOMALY_QUANTILE = 0.95          # 取重建誤差前 5% 當缺陷

# ===== 可重現性 =====
RANDOM_STATE = 42

# ===== Part C：全域門檻 & 量化評估 =====
GLOBAL_BASELINE_IMAGES = 40      # 用多少張圖建立全域誤差基準
GLOBAL_QUANTILE        = 0.995   # 全域 per-patch 誤差分位數 → 像素級異常門檻
IMAGE_SCORE_QUANTILE   = 0.99    # 每張圖的異常分數 = 其 patch 誤差的此分位數
IMAGE_FLAG_QUANTILE    = 0.90    # 影像級:分數超過所有圖此分位數 → 標記為可疑

# 量化評估（合成缺陷注入,產生 ground-truth mask,不需外部標註）
EVAL_IMAGES             = 20     # 評估用影像張數
SYNTH_DEFECTS_PER_IMAGE = 3
SYNTH_DEFECT_MIN        = 8      # 合成缺陷半徑下限 (px)
SYNTH_DEFECT_MAX        = 22     # 合成缺陷半徑上限 (px)


In [ ]:
# 統一設定隨機種子（numpy / random / tensorflow）
import numpy as np, random, os
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
try:
    import tensorflow as tf
    tf.random.set_seed(RANDOM_STATE)
except Exception as e:
    print("TensorFlow 尚未載入,稍後在 Part B 會設定:", e)
print("Seeds set:", RANDOM_STATE)

## 1. 環境設定：Kaggle 憑證與下載 dataset

在 Colab 自動建立 `kaggle.json`（請勿把金鑰寫死在 notebook 中）

In [ ]:
import json, os

# 請將你的 kaggle.json 上傳到 Colab,勿將 key 直接寫在 notebook
# kaggle.json 可從 https://www.kaggle.com/settings -> API -> Create New Token 下載
kaggle_json_path = "/root/.kaggle/kaggle.json"
if not os.path.exists(kaggle_json_path):
    from google.colab import files
    print("請上傳你的 kaggle.json")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open(kaggle_json_path, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(kaggle_json_path, 0o600)
print("Kaggle credentials ready.")

In [ ]:
!pip install kaggle
!kaggle datasets download -d dragonzhang/grainsize-train
!unzip -q grainsize-train.zip -d grainsize_dataset

## 2. 共用函式：載入影像清單、前處理、切 patch

以下 import 與函式在整份 notebook 只定義一次。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 掃描所有影像路徑
image_paths = [p for p in DATA_ROOT.rglob("*") if p.suffix.lower() in IMG_EXT]
print("找到影像數量:", len(image_paths))
image_paths[:5]

In [ ]:
def preprocess_image(path, resize_to=RESIZE_TO):
    """讀灰階 → resize → normalize 到 [0,1] 的 float32。"""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read image: {path}")
    if resize_to is not None:
        img = cv2.resize(img, resize_to, interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0


def extract_patches(img, patch_size=PATCH_SIZE, stride=STRIDE):
    """滑動視窗切 patch,回傳 (patches, coords)。coords 為每個 patch 左上角 (y, x)。"""
    H, W = img.shape
    patches, coords = [], []
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patches.append(img[y:y+patch_size, x:x+patch_size])
            coords.append((y, x))
    return np.stack(patches), np.array(coords)

快速確認：看一張圖與它切出的前幾個 patch。

In [ ]:
img0 = preprocess_image(image_paths[0])
patches0, coords0 = extract_patches(img0)
print("單張圖切出 patch 數:", patches0.shape[0], " 每個尺寸:", patches0.shape[1:])

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(img0, cmap='gray'); axes[0].set_title("image"); axes[0].axis('off')
for i in range(5):
    axes[i+1].imshow(patches0[i], cmap='gray'); axes[i+1].axis('off')
plt.tight_layout(); plt.show()

## 3. 建立整個 dataset 的 patch 庫與 meta

`meta` 是全專案樞紐,形狀 `(N, 3)` = `[img_id, y, x]`,
讓每個 patch 都能回答「來自哪張圖、哪個座標」,之後才能把結果貼回原圖。

In [ ]:
all_patches = []
meta_info   = []
total_patches = 0

for img_id, path in enumerate(image_paths):
    img = preprocess_image(path, resize_to=RESIZE_TO)
    img_patches, coords = extract_patches(img, PATCH_SIZE, STRIDE)

    # 截斷在總量上限
    if total_patches + img_patches.shape[0] > MAX_PATCHES_TOTAL:
        remain = MAX_PATCHES_TOTAL - total_patches
        if remain <= 0:
            break
        img_patches = img_patches[:remain]
        coords      = coords[:remain]

    img_patches = img_patches.astype(np.float16)   # 省記憶體;推論/PCA 時再轉回 float32

    all_patches.append(img_patches)
    img_ids = np.full((coords.shape[0], 1), img_id, dtype=np.int32)
    meta_info.append(np.hstack([img_ids, coords]))  # [img_id, y, x]

    total_patches += img_patches.shape[0]
    if total_patches >= MAX_PATCHES_TOTAL:
        break

# 全域資料集:命名為 dataset_patches / meta,避免與單張圖的 img_patches 混淆
dataset_patches = np.concatenate(all_patches)[..., np.newaxis]   # (N, 64, 64, 1)
meta            = np.concatenate(meta_info)                       # (N, 3)

print("總 patch 數:", dataset_patches.shape[0])
print("dataset_patches 形狀:", dataset_patches.shape, " meta 形狀:", meta.shape)

## Part A — PCA + K-means 微結構分群

**單一套流程**：`StandardScaler → PCA(50) → K-means(K=5)`。
（原始 notebook 有兩套彼此矛盾的分群,此處統一保留有標準化的版本。）

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 為省 RAM,分群只用前 N 個 patch
N = min(dataset_patches.shape[0], PCA_MAX_SAMPLES)
X = dataset_patches[:N].astype(np.float32)          # (N, 64, 64, 1)
X_flat = X.reshape(N, -1)                            # (N, 4096)
print("分群使用 patch 數:", N, " 維度:", X_flat.shape[1])

# 標準化 + PCA
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)
pca_50   = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
X_pca50  = pca_50.fit_transform(X_scaled)
pca_2    = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca2   = pca_2.fit_transform(X_scaled)
print("PCA50:", X_pca50.shape, " PCA2:", X_pca2.shape)
print("PCA50 累積解釋變異:", pca_50.explained_variance_ratio_.sum().round(3))

# K-means（唯一一套分群結果,變數名 clusters）
kmeans   = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(X_pca50)
print("各群大小:", np.bincount(clusters))

2D 散點視覺化分群結果

In [ ]:
plt.figure(figsize=(6, 5))
for k in range(N_CLUSTERS):
    idx = (clusters == k)
    plt.scatter(X_pca2[idx, 0], X_pca2[idx, 1], s=3, alpha=0.5, label=f"C{k}")
plt.legend(markerscale=3); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA 2D + K-means Clusters"); plt.show()

每一群抽 8 個代表 patch 檢視（皆使用同一套 `clusters`）

In [ ]:
def show_cluster_examples(cluster_id, num_examples=8):
    idx = np.where(clusters == cluster_id)[0]
    if len(idx) == 0:
        print(f"Cluster {cluster_id} 沒有樣本"); return
    choose = np.random.choice(idx, size=min(num_examples, len(idx)), replace=False)
    imgs = X[choose, :, :, 0]
    cols = 4; rows = int(np.ceil(len(imgs) / cols))
    plt.figure(figsize=(3*cols, 3*rows))
    for i, im in enumerate(imgs):
        plt.subplot(rows, cols, i+1); plt.imshow(im, cmap='gray'); plt.axis('off')
    plt.suptitle(f"Cluster {cluster_id} examples"); plt.show()

for k in range(N_CLUSTERS):
    show_cluster_examples(k)

### A5. 把 cluster 貼回某張 SEM 圖（cluster map）

用 `meta` 反查座標,將每個 patch 的群編號塗回 512×512 影像。

In [ ]:
def build_cluster_map(img_id, image_paths, meta, clusters, N,
                      resize_to=RESIZE_TO, patch_size=PATCH_SIZE):
    meta_used = meta[:N]                       # 只有前 N 個 patch 有 cluster 標籤
    mask = (meta_used[:, 0] == img_id)
    coords_img   = meta_used[mask][:, 1:3]
    clusters_img = clusters[mask]
    if len(clusters_img) == 0:
        raise ValueError(f"img_id {img_id} 在前 {N} 個 patch 中沒有樣本")

    orig = preprocess_image(image_paths[img_id], resize_to=resize_to)
    H, W = orig.shape
    cluster_map = -1 * np.ones((H, W), dtype=int)
    for (y, x), c in zip(coords_img, clusters_img):
        cluster_map[y:y+patch_size, x:x+patch_size] = c
    return orig, cluster_map


# 從「確定有 patch」的影像中挑一張來畫
valid_ids_A = np.unique(meta[:N, 0])
IMG_ID = int(valid_ids_A[0])

orig, cluster_map = build_cluster_map(IMG_ID, image_paths, meta, clusters, N)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.title("Original (resized)"); plt.imshow(orig, cmap='gray'); plt.axis('off')
plt.subplot(1, 2, 2); plt.title("Cluster Map"); plt.imshow(cluster_map, cmap='tab10', interpolation='nearest'); plt.axis('off')
plt.show()

## Part B — Autoencoder + Reconstruction Error Heatmap（主角）

核心假設：AE 只學會還原「常見（正常）」紋理。重建誤差高的地方 → 可能是缺陷。

### B1. 準備訓練資料（轉回 float32）

In [ ]:
from sklearn.model_selection import train_test_split

X_all = dataset_patches.astype(np.float32)
N_train_use = min(X_all.shape[0], AE_MAX_TRAIN)
X_sub = X_all[:N_train_use]

X_train, X_val = train_test_split(X_sub, test_size=0.1, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Val:", X_val.shape)

### B2. 建立並訓練 Autoencoder

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
tf.random.set_seed(RANDOM_STATE)

inp = layers.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1))
# Encoder
x = layers.Conv2D(32,  (3,3), padding='same', activation='relu')(inp)
x = layers.MaxPool2D((2,2))(x)              # 32x32
x = layers.Conv2D(64,  (3,3), padding='same', activation='relu')(x)
x = layers.MaxPool2D((2,2))(x)              # 16x16
x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
encoded = layers.MaxPool2D((2,2))(x)        # 8x8x128 (bottleneck)
# Decoder
x = layers.Conv2DTranspose(128, (3,3), strides=2, padding='same', activation='relu')(encoded)  # 16x16
x = layers.Conv2DTranspose(64,  (3,3), strides=2, padding='same', activation='relu')(x)         # 32x32
x = layers.Conv2DTranspose(32,  (3,3), strides=2, padding='same', activation='relu')(x)         # 64x64
decoded = layers.Conv2D(1, (3,3), padding='same', activation='sigmoid')(x)

autoencoder = models.Model(inp, decoded)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(AE_LR), loss='mse')
autoencoder.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
]

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=AE_EPOCHS,
    batch_size=AE_BATCH,
    shuffle=True,
    callbacks=callbacks,
)

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('MSE loss'); plt.legend()
plt.title('Autoencoder training'); plt.show()

### B3. 對某張圖計算 reconstruction error heatmap

函式自足（self-contained）：`image_paths` 改為傳入參數,不再偷用全域變數。

In [ ]:
def compute_recon_error_for_image(img_id, patches_all, meta_all, model, image_paths,
                                  patch_size=PATCH_SIZE, resize_to=RESIZE_TO):
    mask = (meta_all[:, 0] == img_id)
    coords      = meta_all[mask][:, 1:3]
    patches_img = patches_all[mask]                  # (M, 64, 64, 1)
    if patches_img.shape[0] == 0:
        raise ValueError(f"img_id {img_id} 沒有對應 patch（可能被 MAX_PATCHES_TOTAL 截斷）")

    recon  = model.predict(patches_img, batch_size=AE_BATCH, verbose=0)
    errors = np.mean((patches_img - recon)**2, axis=(1, 2, 3))   # 每個 patch 一個 MSE

    orig = preprocess_image(image_paths[img_id], resize_to=resize_to)
    H, W = orig.shape
    error_map = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)
    for (y, x), e in zip(coords, errors):
        error_map[y:y+patch_size, x:x+patch_size] += e      # 累加
        count_map[y:y+patch_size, x:x+patch_size] += 1.0    # 計數
    count_map[count_map == 0] = 1.0
    error_map /= count_map                                  # 重疊區取平均,避免格線假影
    return orig, error_map

### B4. 畫 heatmap + 疊圖 + 二值化 defect mask

In [ ]:
# 從「確定有 patch」的影像中挑一張
valid_ids_B = np.unique(meta[:, 0])
IMG_ID = int(valid_ids_B[len(valid_ids_B)//2])   # 取中間一張當示範

orig, error_map = compute_recon_error_for_image(
    img_id=IMG_ID,
    patches_all=dataset_patches.astype(np.float32),
    meta_all=meta,
    model=autoencoder,
    image_paths=image_paths,
)

em_norm = (error_map - error_map.min()) / (error_map.max() - error_map.min() + 1e-8)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.title("Original"); plt.imshow(orig, cmap='gray'); plt.axis('off')
plt.subplot(1, 3, 2); plt.title("Error Map"); plt.imshow(em_norm, cmap='inferno')
plt.colorbar(fraction=0.046, pad=0.04); plt.axis('off')
plt.subplot(1, 3, 3); plt.title("Overlay"); plt.imshow(orig, cmap='gray')
plt.imshow(em_norm, cmap='inferno', alpha=0.5); plt.axis('off')
plt.tight_layout(); plt.show()

二值化缺陷區（取誤差前 5% 當 anomaly,門檻由 `ANOMALY_QUANTILE` 控制）

In [ ]:
threshold   = np.quantile(error_map, ANOMALY_QUANTILE)
defect_mask = (error_map >= threshold)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.title("Original"); plt.imshow(orig, cmap='gray'); plt.axis('off')
plt.subplot(1, 2, 2); plt.title(f"Defect Mask (top {int((1-ANOMALY_QUANTILE)*100)}% error)")
plt.imshow(defect_mask, cmap='gray'); plt.axis('off')
plt.show()

## Part C — 跨圖全域門檻 & 量化評估

Part B 的門檻是「單張圖內」的相對熱區,無法比較不同圖誰更異常。這裡補上兩件事:

1. **全域門檻**：用一批（多為正常的）影像建立 per-patch 誤差基準,得到一個
   跨圖共用的像素級門檻,並替每張圖算一個「異常分數」→ 判定整張圖正常/可疑。
2. **量化評估**：資料集無標註,因此用**合成缺陷注入**產生已知 ground-truth mask,
   計算像素級 **ROC-AUC / IoU** 與影像級 **ROC-AUC**,讓「偵測有效」有數據支撐。


### C0. 可重用的推論函式：任意影像 → error map / per-patch 誤差

可用於資料集內影像,也可用於合成影像(不在 `dataset_patches` 中)。

In [ ]:
def error_map_from_image(img, model, patch_size=PATCH_SIZE, stride=STRIDE):
    """對一張 [0,1] 灰階影像,切 patch → AE 重建 → 回傳 (error_map, per_patch_errors)。"""
    patches, coords = extract_patches(img, patch_size, stride)
    p = patches[..., np.newaxis].astype(np.float32)
    recon  = model.predict(p, batch_size=AE_BATCH, verbose=0)
    errors = np.mean((p - recon) ** 2, axis=(1, 2, 3))     # (M,) 每個 patch 一個 MSE

    H, W = img.shape
    error_map = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)
    for (y, x), e in zip(coords, errors):
        error_map[y:y+patch_size, x:x+patch_size] += e
        count_map[y:y+patch_size, x:x+patch_size] += 1.0
    count_map[count_map == 0] = 1.0
    return error_map / count_map, errors

### C1. 建立全域誤差基準與門檻

In [ ]:
# 取一批影像,匯集所有 per-patch 誤差 → 全域門檻;同時記錄每張圖的異常分數
rng = np.random.default_rng(RANDOM_STATE)
n_base = min(GLOBAL_BASELINE_IMAGES, len(image_paths))
baseline_ids = rng.choice(len(image_paths), size=n_base, replace=False)

pooled_errors = []
image_scores  = []          # (img_id, score)
for img_id in baseline_ids:
    img = preprocess_image(image_paths[int(img_id)], resize_to=RESIZE_TO)
    _, errs = error_map_from_image(img, autoencoder)
    pooled_errors.append(errs)
    image_scores.append((int(img_id), float(np.quantile(errs, IMAGE_SCORE_QUANTILE))))

pooled_errors = np.concatenate(pooled_errors)
GLOBAL_THRESHOLD = float(np.quantile(pooled_errors, GLOBAL_QUANTILE))   # 像素級全域門檻
scores = np.array([s for _, s in image_scores])
IMAGE_LEVEL_THRESHOLD = float(np.quantile(scores, IMAGE_FLAG_QUANTILE)) # 影像級門檻

print(f"全域 per-patch 誤差門檻 (q={GLOBAL_QUANTILE}): {GLOBAL_THRESHOLD:.6f}")
print(f"影像級異常分數門檻  (q={IMAGE_FLAG_QUANTILE}): {IMAGE_LEVEL_THRESHOLD:.6f}")

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.hist(pooled_errors, bins=60); plt.axvline(GLOBAL_THRESHOLD, color='r', ls='--', label='global thr')
plt.title("Pooled per-patch error"); plt.xlabel("MSE"); plt.legend()
plt.subplot(1, 2, 2)
plt.hist(scores, bins=20); plt.axvline(IMAGE_LEVEL_THRESHOLD, color='r', ls='--', label='image thr')
plt.title("Per-image anomaly score"); plt.xlabel("score"); plt.legend()
plt.tight_layout(); plt.show()

用全域門檻替每張圖做「正常 / 可疑」判定(分數最高者最可疑):

In [ ]:
ranked = sorted(image_scores, key=lambda t: t[1], reverse=True)
print("最可疑的前 5 張:")
for img_id, sc in ranked[:5]:
    flag = "SUSPECT" if sc >= IMAGE_LEVEL_THRESHOLD else "normal"
    print(f"  img {img_id:4d}  score={sc:.6f}  -> {flag}")

# 視覺化最可疑那張:同時用「全域門檻」(可跨圖比較) 與原本的單圖門檻
top_id = ranked[0][0]
img = preprocess_image(image_paths[top_id], resize_to=RESIZE_TO)
emap, _ = error_map_from_image(img, autoencoder)
mask_global = emap >= GLOBAL_THRESHOLD

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1); plt.title(f"img {top_id} (most suspect)"); plt.imshow(img, cmap='gray'); plt.axis('off')
plt.subplot(1, 3, 2); plt.title("Error map"); plt.imshow(emap, cmap='inferno'); plt.axis('off')
plt.subplot(1, 3, 3); plt.title("Defect mask (GLOBAL thr)"); plt.imshow(mask_global, cmap='gray'); plt.axis('off')
plt.tight_layout(); plt.show()

### C2. 量化評估：合成缺陷注入 → ROC-AUC / IoU

資料集沒有缺陷標註,無法直接算指標。做法:在正常影像上**注入人工缺陷**
（暗孔洞 / 亮夾雜物）,同時得到精確的 ground-truth mask,再檢驗
重建誤差能否命中這些已知缺陷。這是無監督異常偵測常見的驗證方式。

In [ ]:
def inject_defects(img, n=SYNTH_DEFECTS_PER_IMAGE,
                   r_min=SYNTH_DEFECT_MIN, r_max=SYNTH_DEFECT_MAX, rng=None):
    """在影像上隨機貼上橢圓形缺陷(暗或亮),回傳 (被汙染影像, GT mask)。"""
    rng = rng or np.random.default_rng(RANDOM_STATE)
    out  = img.copy()
    mask = np.zeros(img.shape, dtype=np.uint8)
    H, W = img.shape
    for _ in range(n):
        cx, cy = int(rng.integers(0, W)), int(rng.integers(0, H))
        ax, ay = int(rng.integers(r_min, r_max)), int(rng.integers(r_min, r_max))
        angle  = int(rng.integers(0, 180))
        val    = 0.0 if rng.random() < 0.5 else 1.0        # 暗孔洞 / 亮夾雜物
        blob   = np.zeros(img.shape, dtype=np.uint8)
        cv2.ellipse(blob, (cx, cy), (ax, ay), angle, 0, 360, 1, -1)
        # 缺陷區加一點雜訊,讓它像真實異常紋理
        noisy = np.clip(val + rng.normal(0, 0.05, img.shape).astype(np.float32), 0, 1)
        out[blob == 1] = noisy[blob == 1]
        mask[blob == 1] = 1
    return out, mask.astype(bool)


# demo:看一張注入結果
rng = np.random.default_rng(RANDOM_STATE)
demo = preprocess_image(image_paths[int(baseline_ids[0])], resize_to=RESIZE_TO)
dimg, dmask = inject_defects(demo, rng=rng)
demap, _ = error_map_from_image(dimg, autoencoder)

plt.figure(figsize=(14, 4))
for i, (t, im, cm) in enumerate([("clean", demo, 'gray'), ("with defects", dimg, 'gray'),
                                 ("GT mask", dmask, 'gray'), ("error map", demap, 'inferno')]):
    plt.subplot(1, 4, i+1); plt.title(t); plt.imshow(im, cmap=cm); plt.axis('off')
plt.tight_layout(); plt.show()

在多張影像上計算指標:像素級 ROC-AUC、IoU(用全域門檻),以及影像級 ROC-AUC。

In [ ]:
from sklearn.metrics import roc_auc_score

def iou(pred_mask, gt_mask):
    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return inter / union if union > 0 else np.nan

rng = np.random.default_rng(RANDOM_STATE + 1)
eval_ids = rng.choice(len(image_paths), size=min(EVAL_IMAGES, len(image_paths)), replace=False)

pixel_aucs, ious = [], []
img_scores_normal, img_scores_defect = [], []

for img_id in eval_ids:
    clean = preprocess_image(image_paths[int(img_id)], resize_to=RESIZE_TO)
    dirty, gt = inject_defects(clean, rng=rng)

    emap_d, errs_d = error_map_from_image(dirty, autoencoder)
    _,      errs_c = error_map_from_image(clean, autoencoder)

    # 像素級:誤差值當分數 vs GT mask
    if gt.any() and (~gt).any():
        pixel_aucs.append(roc_auc_score(gt.ravel(), emap_d.ravel()))
        ious.append(iou(emap_d >= GLOBAL_THRESHOLD, gt))

    # 影像級分數(同 C1 的定義)
    img_scores_normal.append(np.quantile(errs_c, IMAGE_SCORE_QUANTILE))
    img_scores_defect.append(np.quantile(errs_d, IMAGE_SCORE_QUANTILE))

# 影像級 AUC:正常(0) vs 注入缺陷(1)
y_true  = np.r_[np.zeros(len(img_scores_normal)), np.ones(len(img_scores_defect))]
y_score = np.r_[img_scores_normal, img_scores_defect]
image_auc = roc_auc_score(y_true, y_score)

print(f"評估影像數: {len(eval_ids)}")
print(f"像素級 ROC-AUC : {np.mean(pixel_aucs):.3f}  (±{np.std(pixel_aucs):.3f})")
print(f"像素級 IoU@全域門檻: {np.nanmean(ious):.3f}")
print(f"影像級 ROC-AUC : {image_auc:.3f}   (正常 vs 合成缺陷)")

> **判讀**：像素級 AUC 越接近 1,代表誤差圖越能把缺陷像素排在前面;
> IoU 反映在全域門檻下預測缺陷區與真實缺陷區的重疊程度(受門檻鬆緊影響);
> 影像級 AUC 反映「用一個分數區分正常/異常整張圖」的能力。
>
> 注意:合成缺陷與真實缺陷的分布未必相同,此評估用於**相對驗證與調參**,
> 不能完全等同於真實缺陷上的表現。若日後取得少量真實標註,可把
> `inject_defects` 換成真實 mask,程式其餘不變。